# LoraRetriever: Input-Aware LoRA Retrieval and Composition
### Reproduction Guide — ACL 2024 Findings (arXiv 2402.09997)

---

## 🧠 How the Method Works — Deep Dive

### The Core Problem
In the real world, a deployed LLM receives **mixed task inputs**: NLI, summarization, QA, translation, etc. You may have a *library* of task-specific LoRA adapters (one per task, trained separately). The challenge is: **given a new input, which LoRA(s) should you activate, and how should you combine them?**

### LoraRetriever's 3-Stage Retrieve-then-Compose Framework

```
                  ┌──────────────────────────────────────────────┐
  New Input ─────►│  Stage 1: LoRA Retrieval (embedding search)  │
                  └──────────────────┬───────────────────────────┘
                                     │ top-k LoRAs
                  ┌──────────────────▼───────────────────────────┐
                  │  Stage 2: LoRA Composition (3 strategies)    │
                  └──────────────────┬───────────────────────────┘
                                     │ composed adapter
                  ┌──────────────────▼───────────────────────────┐
                  │  Stage 3: Batch Inference (hetero requests)  │
                  └──────────────────────────────────────────────┘
```

---

### Stage 1 — LoRA Retrieval via Instruction Embeddings

**Goal:** Represent each LoRA as a vector, then find the most similar LoRAs to the query input.

**LoRA Representation:**  
Each task-specific LoRA is characterised by randomly sampling ~20 examples from its training set. These examples are then encoded with an instruction-tuned sentence encoder (`instructor-xl`) using the instruction prompt:  
> `"Represent the sentence for similar task retrieval:"`  

The 20 embedding vectors are **mean-pooled** into a single vector per LoRA. This creates a lightweight LoRA index in embedding space.

**Query Encoding:**  
At inference time, the new input (instruction + input text) is encoded the same way with the same instruction prefix.

**Retrieval:**  
Cosine similarity between the query embedding and all LoRA index vectors → pick **top-k** (k=3 in the paper).

**Retriever Fine-Tuning (for generalization to unseen LoRAs):**  
To generalise to LoRAs *not seen at training time*, the encoder is fine-tuned with contrastive learning:  
- **Positive pairs:** Two examples from the same LoRA task  
- **Negative pairs:** Examples from different LoRA tasks  
- Only ~40% of available tasks are used for training — the rest simulate *unseen LoRAs* at test time.

---

### Stage 2 — LoRA Composition (3 Strategies)

Given the top-k retrieved LoRAs `{A₁B₁, A₂B₂, ..., AₖBₖ}` (where each LoRA has down-projection `A` and up-projection `B`), three composition strategies are compared:

**1. Parameter Fusion (Weight-space merging)**  
Each LoRA weight delta `ΔW = A·B` is merged into a single delta via **weighted average**:
```
ΔW_fused = Σᵢ wᵢ · (Aᵢ · Bᵢ)
```
Weights `wᵢ` can be uniform (equal weight) or similarity-based (softmax of cosine similarities).  
This is single-pass and most efficient — one forward pass through the merged model.

**2. Output Mixture (Activation-space ensembling)**  
Each LoRA is applied *separately*, and their output contributions are merged:  
```
output = W₀·x + Σᵢ wᵢ · (Aᵢ·Bᵢ·x)
```
This doesn't require merging weights — all adapters remain separate and are added together on the fly during the forward pass. More expressive but more memory-intensive.

**3. Top-1 Selection (No composition)**  
Simply use the single highest-ranked retrieved LoRA with no mixing. Acts as a strong ablation baseline.

---

### Stage 3 — Efficient Batch Inference

In real deployments, a batch of inputs may each require *different* LoRA combinations. Naively this would require separate forward passes. LoraRetriever instead **groups requests by their LoRA assignment** and pads within groups to support batching — a form of dynamic batching for heterogeneous LoRA mixtures.

---

### How LoraRetriever Differs from LoraHub

| | LoraHub | LoraRetriever |
|---|---|---|  
| LoRA selection | Gradient-free optimisation per task | Embedding retrieval per input |
| Granularity | Per task (same mix for all inputs in a task) | Per input (each prompt gets its own mix) |
| Training needed | No (gradient-free adapt) | Optional retriever fine-tuning |
| Speed | Slower (iterative optimisation) | Fast (single retrieval step) |
| Dynamic LoRA pool | No | Yes |

---

## 📦 0. Environment Setup

In [ ]:
# Run this cell once to install all dependencies
# Requires Python 3.10+ and a CUDA-capable GPU (A100/A10G recommended for Llama-2-7b)

!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install transformers==4.34.0
!pip install peft==0.6.0          # official peft; LoraRetriever ships a custom fork — see note below
!pip install InstructorEmbedding   # for instructor-xl retriever
!pip install sentence-transformers
!pip install datasets
!pip install accelerate
!pip install scipy numpy tqdm
!pip install lorahub               # for reference to LoraHub composition utilities

# LoraRetriever uses a PATCHED version of peft that supports multi-LoRA loading
# Clone the repo and install its bundled peft:
!git clone https://github.com/StyxXuan/LoraRetriever.git
%cd LoraRetriever
!pip install -e peft/
!pip install -r requirements.txt

In [ ]:
import os, json, random
import numpy as np
import torch
from pathlib import Path

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 📥 1. Download Resources

LoraRetriever releases two sets of LoRAs on HuggingFace:
- **Llama-2-7b LoRAs** (lighter, recommended for reproduction): `Styxxxx/loraretriever-llama2-7b-loras-*`
- **Llama-2-13b LoRAs**: requires more VRAM

Each LoRA covers one of 54+ NLP tasks across categories: NLI, QA, sentiment analysis, summarization, closed-book QA, coreference, reading comprehension, etc.

In [ ]:
from huggingface_hub import snapshot_download, list_models

# ---- Config ----
BASE_MODEL_ID  = "meta-llama/Llama-2-7b-hf"   # needs HF access token
LORA_COLLECTION = "Styxxxx"                    # HuggingFace org with released LoRAs
LORA_SAVE_DIR   = Path("./lora_modules")
LORA_SAVE_DIR.mkdir(exist_ok=True)

# List all 7b lora repos released by authors
all_repos = list(list_models(author=LORA_COLLECTION, search="llama2-7b"))
lora_repo_ids = [m.modelId for m in all_repos]
print(f"Found {len(lora_repo_ids)} LoRA modules:")
for r in lora_repo_ids[:10]:
    print(" ", r)

In [ ]:
# Download a subset of LoRAs for quick experimentation (the paper uses ~54 tasks)
# Each LoRA is ~30-80 MB (only adapter_model.bin + adapter_config.json)

TASK_SUBSET = lora_repo_ids[:20]   # reduce to 5-10 if storage is tight

for repo_id in TASK_SUBSET:
    task_name = repo_id.split("/")[-1]  # e.g. "loraretriever-llama2-7b-sst2"
    save_path = LORA_SAVE_DIR / task_name
    if not save_path.exists():
        print(f"Downloading {task_name}...")
        snapshot_download(repo_id=repo_id, local_dir=str(save_path))
    else:
        print(f"Already cached: {task_name}")

print(f"\nTotal LoRAs on disk: {len(list(LORA_SAVE_DIR.iterdir()))}")

## 🗃️ 2. Build the LoRA Index (Retriever Side)

**Key insight:** Each LoRA is *represented* by the average `instructor-xl` embedding of ~20 randomly-sampled training examples from that LoRA's task.

The retriever uses the instruction prefix `"Represent the sentence for similar task retrieval:"` — this is the instruction-tuned prompting style from the Instructor paper (EMNLP 2023).

The result is a dictionary: `{task_name → embedding_vector (dim=768)}`

In [ ]:
from InstructorEmbedding import INSTRUCTOR

# Load instructor-xl (or instructor-large for lighter footprint)
RETRIEVER_MODEL = "hkunlp/instructor-xl"
retriever = INSTRUCTOR(RETRIEVER_MODEL)
retriever.eval()
print("Retriever loaded:", RETRIEVER_MODEL)

In [ ]:
# ---- Load few-shot representative examples per LoRA ----
# These come from the dataset/ directory in the LoraRetriever repo
# Format: each task has a .json file with list of {"input": ..., "output": ...} dicts

DATA_DIR = Path("./dataset")
INSTRUCTION_PREFIX = "Represent the sentence for similar task retrieval:"
N_SAMPLES_PER_LORA = 20

def load_task_samples(task_json_path, n=N_SAMPLES_PER_LORA):
    """Load n examples from a task JSON file."""
    with open(task_json_path) as f:
        data = json.load(f)
    samples = random.sample(data, min(n, len(data)))
    return [s["input"] for s in samples]  # use input text only

def embed_texts(texts, instruction=INSTRUCTION_PREFIX):
    """Encode a list of texts with the instructor model."""
    pairs = [[instruction, t] for t in texts]
    with torch.no_grad():
        embeddings = retriever.encode(pairs, batch_size=8, show_progress_bar=False)
    return embeddings  # shape: (n_texts, 768)

def get_lora_embedding(task_json_path):
    """Get the mean embedding for a LoRA from its task samples."""
    texts = load_task_samples(task_json_path)
    embeddings = embed_texts(texts)
    return embeddings.mean(axis=0)  # mean-pool → single 768-dim vector

In [ ]:
# Build the LoRA index: {task_name -> (embedding, lora_path)}
# We match each LoRA directory to its corresponding training data JSON

lora_index = {}  # task_name -> {"embedding": np.array, "lora_path": str}

for lora_dir in sorted(LORA_SAVE_DIR.iterdir()):
    task_name = lora_dir.name  # e.g. "loraretriever-llama2-7b-sst2"
    # Extract short task identifier (last part after final dash segments)
    short_name = task_name.replace("loraretriever-llama2-7b-", "")
    
    # Look for matching dataset file
    task_data_file = DATA_DIR / f"{short_name}.json"
    if not task_data_file.exists():
        # Try to find closest match
        candidates = list(DATA_DIR.glob(f"*{short_name}*.json"))
        if not candidates:
            print(f"  WARNING: No dataset found for {short_name}, skipping")
            continue
        task_data_file = candidates[0]
    
    print(f"  Indexing LoRA: {short_name}")
    embedding = get_lora_embedding(task_data_file)
    lora_index[short_name] = {
        "embedding": embedding,
        "lora_path": str(lora_dir)
    }

print(f"\nIndexed {len(lora_index)} LoRAs")

# Stack embeddings for efficient cosine search
index_keys   = list(lora_index.keys())
index_matrix = np.stack([lora_index[k]["embedding"] for k in index_keys])  # (N_loras, 768)
print(f"Index matrix shape: {index_matrix.shape}")

## 🔍 3. Input-Aware LoRA Retrieval

For each incoming query, we embed the instruction+input with the same `instructor-xl` encoder and compute cosine similarity against the LoRA index matrix to find the top-k most relevant LoRAs.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_top_k_loras(query_text, k=3, instruction=INSTRUCTION_PREFIX):
    """
    Given a query input string, retrieve the top-k most relevant LoRAs.
    Returns: list of (task_name, similarity_score, lora_path) sorted by descending score
    """
    # 1. Embed the query
    query_emb = embed_texts([query_text], instruction)  # (1, 768)
    
    # 2. Cosine similarity against all LoRA index embeddings
    sims = cosine_similarity(query_emb, index_matrix)[0]  # (N_loras,)
    
    # 3. Sort descending and take top-k
    top_k_idx = np.argsort(sims)[::-1][:k]
    
    results = []
    for idx in top_k_idx:
        task = index_keys[idx]
        results.append({
            "task": task,
            "score": float(sims[idx]),
            "lora_path": lora_index[task]["lora_path"]
        })
    return results


# ---- Quick test ----
test_queries = [
    "Classify the sentiment: 'The movie was absolutely fantastic!'",
    "Summarize the following article in one sentence: NASA announced...",
    "Does the hypothesis follow from the premise? Premise: All dogs are animals. Hypothesis: Some animals are dogs."
]

for q in test_queries:
    top_loras = retrieve_top_k_loras(q, k=3)
    print(f"\nQuery: {q[:60]}...")
    for r in top_loras:
        print(f"  [{r['score']:.4f}] {r['task']}")

## 🔀 4. LoRA Composition Strategies

Three strategies are implemented. We'll load the base Llama-2-7b model once, then apply each strategy.

In [ ]:
import copy
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, PeftConfig

# ---- Load base model once (keep frozen) ----
print("Loading base model (this may take a few minutes)...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",  # auto-distributes across GPUs
    load_in_8bit=False  # set True to halve VRAM if needed
)
base_model.eval()
print("Base model loaded")

In [ ]:
def load_lora_weights(lora_path):
    """
    Load LoRA adapter tensors (A and B matrices) from disk.
    Returns: dict mapping layer_name -> {lora_A: tensor, lora_B: tensor, scaling: float}
    """
    import safetensors.torch as sf
    
    weights_file = os.path.join(lora_path, "adapter_model.safetensors")
    if not os.path.exists(weights_file):
        weights_file = os.path.join(lora_path, "adapter_model.bin")
        state_dict = torch.load(weights_file, map_location="cpu")
    else:
        state_dict = sf.load_file(weights_file)
    
    config = PeftConfig.from_pretrained(lora_path)
    lora_alpha = config.lora_alpha
    lora_r     = config.r
    scaling    = lora_alpha / lora_r
    
    # Group by layer: {"model.layers.0.self_attn.q_proj": {"lora_A": ..., "lora_B": ...}}
    layers = {}
    for key, val in state_dict.items():
        # key format: "base_model.model.model.layers.X.self_attn.q_proj.lora_A.weight"
        if "lora_A" in key:
            layer = key.replace(".lora_A.weight", "").replace("base_model.model.", "")
            layers.setdefault(layer, {})["lora_A"] = val.float()
        elif "lora_B" in key:
            layer = key.replace(".lora_B.weight", "").replace("base_model.model.", "")
            layers.setdefault(layer, {})["lora_B"] = val.float()
    
    for layer in layers:
        layers[layer]["scaling"] = scaling
    
    return layers


def compute_delta_W(lora_weights):
    """
    Compute ΔW = scaling * B @ A for each layer.
    Returns: dict layer_name -> delta_W tensor
    """
    delta_Ws = {}
    for layer, w in lora_weights.items():
        # B: (out, r), A: (r, in)  →  B @ A: (out, in)
        delta_W = w["scaling"] * (w["lora_B"] @ w["lora_A"])
        delta_Ws[layer] = delta_W
    return delta_Ws

In [ ]:
# ================================================================
# STRATEGY 1: Top-1 Selection
# Simply load the single best-matching LoRA. No composition.
# ================================================================

def compose_top1(base_model, retrieved_loras):
    """Load only the top-1 retrieved LoRA adapter."""
    best = retrieved_loras[0]
    print(f"  Using top-1 LoRA: {best['task']} (score={best['score']:.4f})")
    model = PeftModel.from_pretrained(base_model, best["lora_path"])
    return model


# ================================================================
# STRATEGY 2: Parameter Fusion (weight-space merging)
# Weighted average of ΔW matrices, then add to base weights.
# ================================================================

def compose_parameter_fusion(base_model, retrieved_loras, weight_by_score=True):
    """
    Merge top-k LoRAs via weighted average of their ΔW matrices.
    
    If weight_by_score=True, uses softmax of cosine similarities as weights.
    Otherwise uses uniform weights (1/k each).
    """
    k = len(retrieved_loras)
    
    # Compute fusion weights
    if weight_by_score:
        scores = np.array([r["score"] for r in retrieved_loras])
        # Apply temperature-scaled softmax
        T = 0.1
        weights = np.exp(scores / T) / np.exp(scores / T).sum()
    else:
        weights = np.ones(k) / k
    
    print(f"  Fusion weights: {dict(zip([r['task'] for r in retrieved_loras], weights.round(4)))}")
    
    # Load all LoRA ΔW tensors
    all_delta_Ws = []
    for r in retrieved_loras:
        lora_weights = load_lora_weights(r["lora_path"])
        delta_Ws = compute_delta_W(lora_weights)
        all_delta_Ws.append(delta_Ws)
    
    # Compute fused ΔW: Σᵢ wᵢ · ΔWᵢ
    fused_delta_W = {}
    reference_layers = all_delta_Ws[0].keys()
    for layer in reference_layers:
        fused_delta_W[layer] = sum(
            weights[i] * all_delta_Ws[i][layer]
            for i in range(k)
            if layer in all_delta_Ws[i]
        )
    
    # Inject fused ΔW directly into the base model's weights
    model = copy.deepcopy(base_model)
    with torch.no_grad():
        for name, param in model.named_parameters():
            # Strip common prefix variations to match layer keys
            clean = name.replace(".weight", "")
            if clean in fused_delta_W:
                param.data += fused_delta_W[clean].to(param.device, param.dtype)
    
    return model


# ================================================================
# STRATEGY 3: Output Mixture (activation-space ensembling)
# Keep all LoRAs separate; blend their output activations at runtime.
# ================================================================

def compose_output_mixture(base_model, retrieved_loras, weight_by_score=True):
    """
    Apply each LoRA separately and mix their output contributions.
    This is implemented via PeftModel by loading the first LoRA then adding others.
    The custom peft/ fork in LoraRetriever adds direct support for this.
    """
    k = len(retrieved_loras)
    if weight_by_score:
        scores = np.array([r["score"] for r in retrieved_loras])
        T = 0.1
        weights = np.exp(scores / T) / np.exp(scores / T).sum()
    else:
        weights = np.ones(k) / k
    
    print(f"  Output mixture weights: {dict(zip([r['task'] for r in retrieved_loras], weights.round(4)))}")
    
    # Load first LoRA as the PEFT base
    model = PeftModel.from_pretrained(
        base_model,
        retrieved_loras[0]["lora_path"],
        adapter_name="lora_0"
    )
    
    # Load additional LoRAs with unique adapter names
    for i, r in enumerate(retrieved_loras[1:], 1):
        model.load_adapter(r["lora_path"], adapter_name=f"lora_{i}")
    
    # The LoraRetriever custom PEFT fork supports weighted combination
    # via set_adapter with a weight dict (or we can manual-mix in the forward hook)
    # Here we demonstrate manual output mixing via a forward hook
    adapter_names = [f"lora_{i}" for i in range(k)]
    
    # Store weights on model for access in hooks
    model._mixture_weights = {name: float(weights[i]) for i, name in enumerate(adapter_names)}
    model._adapter_names   = adapter_names
    
    print(f"  Loaded {k} adapters for output mixture: {adapter_names}")
    return model, weights

## 🤖 5. Inference Pipeline

Full end-to-end pipeline: given a raw instruction+input, retrieve LoRAs, compose, and generate.

In [ ]:
# ---- Alpaca-style prompt template (matches LoraRetriever training format) ----

ALPACA_TEMPLATE = """\
Below is an instruction that describes a task, paired with an input that provides further context. \
Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:"""

ALPACA_TEMPLATE_NO_INPUT = """\
Below is an instruction that describes a task. \
Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Response:"""

def format_prompt(instruction, input_text=""):
    if input_text.strip():
        return ALPACA_TEMPLATE.format(instruction=instruction, input=input_text)
    else:
        return ALPACA_TEMPLATE_NO_INPUT.format(instruction=instruction)


def generate_response(model, tokenizer, prompt, max_new_tokens=128):
    """Tokenize prompt and generate response."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(model.device if hasattr(model, 'device') else 'cuda') for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # greedy for reproducibility
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode only the newly generated tokens
    input_len = inputs["input_ids"].shape[1]
    generated = outputs[0][input_len:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


def loraretriever_pipeline(instruction, input_text="", k=3, strategy="top1"):
    """
    Full LoraRetriever inference pipeline.
    
    Args:
        instruction: Task instruction string
        input_text: Input text (optional)
        k: Number of LoRAs to retrieve
        strategy: one of "top1", "parameter_fusion", "output_mixture"
    
    Returns:
        dict with retrieved LoRAs and generated response
    """
    # Format query for retrieval (instruction + input combined)
    query = f"{instruction} {input_text}".strip()
    
    # Step 1: Retrieve top-k LoRAs
    print(f"\n{'='*60}")
    print(f"Query: {query[:80]}")
    retrieved = retrieve_top_k_loras(query, k=k)
    print(f"Retrieved LoRAs:")
    for r in retrieved:
        print(f"  {r['task']:40s} sim={r['score']:.4f}")
    
    # Step 2: Compose LoRAs
    print(f"Composing with strategy: {strategy}")
    if strategy == "top1":
        model = compose_top1(base_model, retrieved)
    elif strategy == "parameter_fusion":
        model = compose_parameter_fusion(base_model, retrieved)
    elif strategy == "output_mixture":
        model, _ = compose_output_mixture(base_model, retrieved)
        # For output_mixture with k>1, you need the patched peft; 
        # as a simplified version, activate only the top-1 here:
        model.set_adapter("lora_0")
    else:
        raise ValueError(f"Unknown strategy: {strategy}")
    
    # Step 3: Format prompt and generate
    prompt = format_prompt(instruction, input_text)
    response = generate_response(model, tokenizer, prompt)
    
    print(f"Response: {response}")
    return {
        "retrieved_loras": retrieved,
        "strategy": strategy,
        "response": response
    }

In [ ]:
# ---- Run the full pipeline on example queries ----

examples = [
    {
        "instruction": "Classify the sentiment of the following review.",
        "input": "The acting was brilliant and the storyline kept me hooked throughout.",
    },
    {
        "instruction": "Does the hypothesis logically follow from the premise?",
        "input": "Premise: The cat sat on the mat. Hypothesis: There is a cat somewhere.",
    },
    {
        "instruction": "Answer the following question about the given passage.",
        "input": "Passage: The Eiffel Tower was built in 1889. Question: When was the Eiffel Tower built?",
    }
]

for ex in examples:
    result = loraretriever_pipeline(
        instruction=ex["instruction"],
        input_text=ex["input"],
        k=3,
        strategy="top1"   # Start with top1; change to "parameter_fusion" to compare
    )

## 🎓 6. (Optional) Fine-Tuning the Retriever for Better Generalization

The paper shows that fine-tuning `instructor-xl` with contrastive learning on **40% of the tasks** improves retrieval accuracy on *unseen* LoRAs. Here's how to set that up.

- Positive pairs: two examples from the **same task/LoRA**
- Negative pairs: examples from **different tasks**
- Loss: Multiple Negatives Ranking Loss (MNR) — standard for bi-encoder training

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

# ---- Build contrastive training dataset ----

def build_contrastive_dataset(data_dir, task_names, n_per_task=50):
    """
    Build (anchor, positive, negative) triplets for retriever fine-tuning.
    anchor   = example from task A
    positive = another example from task A
    negative = example from task B (in-batch negatives handled by MNR loss)
    """
    all_task_data = {}
    for task in task_names:
        fpath = Path(data_dir) / f"{task}.json"
        if fpath.exists():
            with open(fpath) as f:
                data = json.load(f)
            all_task_data[task] = [d["input"] for d in data[:n_per_task]]
    
    examples = []
    task_list = list(all_task_data.keys())
    
    for task, texts in all_task_data.items():
        # Pair consecutive examples from the same task as (anchor, positive)
        for i in range(0, len(texts) - 1, 2):
            examples.append(InputExample(texts=[texts[i], texts[i+1]]))
    
    print(f"Built {len(examples)} training pairs from {len(task_list)} tasks")
    return examples


# Use only 40% of tasks for training (simulate unseen LoRAs at test time)
all_tasks  = [f.stem for f in Path(DATA_DIR).glob("*.json")]
train_size = int(0.4 * len(all_tasks))
random.shuffle(all_tasks)
train_tasks = all_tasks[:train_size]
test_tasks  = all_tasks[train_size:]

print(f"Train tasks: {len(train_tasks)}, Test/unseen tasks: {len(test_tasks)}")

train_examples = build_contrastive_dataset(DATA_DIR, train_tasks)

In [ ]:
# ---- Fine-tune the retriever ----
# We use SentenceTransformers' MNR loss, which is equivalent to the contrastive loss in the paper

# Load instructor-xl as a SentenceTransformer for fine-tuning
st_retriever = SentenceTransformer("hkunlp/instructor-xl")

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
train_loss = losses.MultipleNegativesRankingLoss(st_retriever)

# Train for 1 epoch (paper trains on 40% tasks, ~20 samples each)
st_retriever.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=1,
    warmup_steps=50,
    output_path="./retriever_finetuned",
    show_progress_bar=True
)

print("Retriever fine-tuning complete. Saved to ./retriever_finetuned")

In [ ]:
# ---- Reload fine-tuned retriever and rebuild index ----
retriever = INSTRUCTOR("./retriever_finetuned")
retriever.eval()

# Rebuild the LoRA index with fine-tuned embeddings
lora_index_ft  = {}
for task, info in lora_index.items():
    task_data_file = DATA_DIR / f"{task}.json"
    if task_data_file.exists():
        embedding = get_lora_embedding(task_data_file)
        lora_index_ft[task] = {"embedding": embedding, "lora_path": info["lora_path"]}

index_keys_ft   = list(lora_index_ft.keys())
index_matrix_ft = np.stack([lora_index_ft[k]["embedding"] for k in index_keys_ft])
print(f"Rebuilt fine-tuned index: {index_matrix_ft.shape}")

## 📊 7. Evaluation on LoraRetriever Benchmark

The benchmark covers 54 tasks across clusters: NLI, QA, Sentiment, Reading Comprehension, etc.  
Metrics: **Accuracy** (classification tasks), **RougeL** (generation tasks).

In [ ]:
from datasets import load_dataset
from rouge_score import rouge_scorer
import re

def normalize_answer(s):
    """Lower text and remove punctuation, articles and extra whitespace."""
    s = s.lower().strip()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = re.sub(r'[^\w\s]', '', s)
    s = ' '.join(s.split())
    return s

def compute_exact_match(prediction, ground_truth):
    return normalize_answer(prediction) == normalize_answer(ground_truth)

def compute_rougeL(prediction, ground_truth):
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    scores = scorer.score(ground_truth, prediction)
    return scores['rougeL'].fmeasure


def evaluate_task(task_name, eval_examples, strategy="top1", k=3, metric="accuracy"):
    """
    Evaluate LoraRetriever on a single task's eval set.
    """
    predictions, references = [], []
    
    for ex in eval_examples:
        instruction = ex.get("instruction", "")
        input_text  = ex.get("input", "")
        label       = ex.get("output", "")
        
        query = f"{instruction} {input_text}".strip()
        retrieved = retrieve_top_k_loras(query, k=k)
        
        if strategy == "top1":
            model = compose_top1(base_model, retrieved)
        elif strategy == "parameter_fusion":
            model = compose_parameter_fusion(base_model, retrieved)
        
        prompt = format_prompt(instruction, input_text)
        pred   = generate_response(model, tokenizer, prompt)
        
        predictions.append(pred)
        references.append(label)
    
    if metric == "accuracy":
        correct = sum(compute_exact_match(p, r) for p, r in zip(predictions, references))
        score   = correct / len(predictions)
    elif metric == "rougeL":
        score = np.mean([compute_rougeL(p, r) for p, r in zip(predictions, references)])
    
    print(f"  Task: {task_name:30s}  {metric}: {score:.4f}  (n={len(eval_examples)})")
    return score, predictions, references


print("Evaluation functions ready")

In [ ]:
# ---- Evaluate on the LoraRetriever test set ----
# The eval data lives in ./data/ directory of the cloned repo
# Each file: list of {"instruction": ..., "input": ..., "output": ...}

EVAL_DIR = Path("./data")
results_table = []

# Evaluate the first 3 tasks as a quick sanity check
# (Run all 54 tasks for full paper reproduction — takes ~hours)
eval_tasks = sorted(EVAL_DIR.glob("*.json"))[:3]

for task_path in eval_tasks:
    task_name = task_path.stem
    with open(task_path) as f:
        eval_data = json.load(f)
    
    # Use up to 100 eval examples per task (paper uses full set)
    eval_examples = eval_data[:100]
    
    for strategy in ["top1", "parameter_fusion"]:
        metric = "accuracy"  # swap to "rougeL" for generation tasks
        score, preds, refs = evaluate_task(
            task_name, eval_examples, strategy=strategy, k=3, metric=metric
        )
        results_table.append({
            "task": task_name,
            "strategy": strategy,
            "metric": metric,
            "score": score
        })

print("\n===== Results Summary =====")
print(f"{'Task':<35} {'Strategy':<20} {'Score':<10}")
print("-" * 65)
for r in results_table:
    print(f"{r['task']:<35} {r['strategy']:<20} {r['score']:.4f}")

In [ ]:
# ---- Compare strategies side-by-side ----
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame(results_table)
pivot = df.pivot(index="task", columns="strategy", values="score")
print(pivot)

pivot.plot(kind="bar", figsize=(12, 5), title="LoraRetriever: Strategy Comparison by Task")
plt.ylabel("Score")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("strategy_comparison.png", dpi=150)
plt.show()

## 🔁 8. Using eval_all.sh from the Official Repo

The official repo ships `eval_all.sh` which runs the full evaluation suite via `main.py`. Here's how to run it and interpret the output.

In [ ]:
# Run the official evaluation script (from inside the cloned LoraRetriever directory)
# This evaluates all 54 tasks with all strategies as reported in the paper

# First inspect what eval_all.sh does:
!cat eval_all.sh | head -40

In [ ]:
# Inspect main.py to understand arguments
!python main.py --help 2>/dev/null || cat main.py | head -80

In [ ]:
# Run evaluation for a single task manually via the official script
# Adjust --model_name, --lora_path, --task_name as needed

!python main.py \
    --base_model meta-llama/Llama-2-7b-hf \
    --lora_dir ./lora_modules \
    --data_dir ./data \
    --output_dir ./results \
    --composition_strategy parameter_fusion \
    --top_k 3 \
    --retriever_model hkunlp/instructor-xl \
    --task sst2

In [ ]:
# Summarize results after running eval_all.sh
!python summarize_results.py --results_dir ./results

## 🚀 9. Reference: LoraHub Comparison

LoraHub optimises composition weights via a **gradient-free evolutionary algorithm** (CMA-ES from `nevergrad`) per task using a handful of examples. Here's how to use it to compare directly with LoraRetriever.

In [ ]:
from lorahub.algorithm import lorahub_learning, lorahub_inference
from lorahub.constant import LORA_MODULE_NAMES

# LoraHub works on Flan-T5 LoRAs (different from LoraRetriever which uses Llama-2)
# The comparison in the paper uses the same base model and LoRA pool
# Here we demonstrate the LoraHub API for reference

def run_lorahub_on_task(task_examples, lora_candidates=None, max_steps=40):
    """
    Use LoraHub's gradient-free composition on a few examples.
    task_examples: list of {"input": ..., "output": ...}
    """
    if lora_candidates is None:
        random.seed(42)
        lora_candidates = random.sample(LORA_MODULE_NAMES, 20)
    
    inputs  = [e["input"] for e in task_examples]
    outputs = [e["output"] for e in task_examples]
    
    # LoraHub learns weights over the candidate pool via CMA-ES
    weights, model, tokenizer = lorahub_learning(
        lora_module_list=lora_candidates,
        example_inputs=inputs,
        example_outputs=outputs,
        max_inference_step=max_steps,
        batch_size=2
    )
    print("LoraHub learned weights:", dict(zip(lora_candidates, weights.round(3))))
    return model, tokenizer, weights


# Example: Use 5 examples to learn composition weights for a sentiment task
few_shot = [
    {"input": "Classify sentiment: Great product!", "output": "positive"},
    {"input": "Classify sentiment: Terrible experience.", "output": "negative"},
    {"input": "Classify sentiment: It was okay.", "output": "neutral"},
    {"input": "Classify sentiment: Loved every minute!", "output": "positive"},
    {"input": "Classify sentiment: Would not recommend.", "output": "negative"},
]

# Uncomment to run (requires Flan-T5 LoRAs from LoraHub):
# model_lh, tok_lh, weights_lh = run_lorahub_on_task(few_shot)

## 📋 10. Reproduction Checklist

| Step | What to do | Notes |
|------|------------|-------|
| ✅ 0 | Install environment | `pip install -e peft/` uses LoraRetriever's patched peft |
| ✅ 1 | Download LoRAs from HuggingFace | `Styxxxx` org, Llama-2-7b or 13b |
| ✅ 2 | Build retriever index | Mean-pool instructor-xl embeddings of 20 samples/LoRA |
| ✅ 3 | (Optional) Fine-tune retriever | Contrastive training on 40% tasks, MNR loss |
| ✅ 4 | Retrieve top-k LoRAs per input | Cosine similarity, k=3 |
| ✅ 5 | Compose: Top-1, Parameter Fusion, or Output Mixture | See paper Table 2 |
| ✅ 6 | Run eval_all.sh for full benchmark | 54 tasks, 7 task clusters |
| ✅ 7 | Compare against LoraHub | LoraHub = per-task gradient-free weight optimisation |

### Key Hyperparameters (from paper)

| Parameter | Value |
|-----------|-------|
| Number retrieved LoRAs (k) | 3 |
| Retriever model | `hkunlp/instructor-xl` |
| Retriever training tasks | 40% of all tasks |
| Retriever training samples/task | 20 |
| LoRA rank (r) | 8 |
| LoRA alpha | 16 |
| Base model | Llama-2-7b / Llama-2-13b |
| Evaluation tasks | 54 across 7 clusters |

In [ ]:
# ---- Final: Save index for reuse ----
import pickle

index_save = {
    "keys": index_keys,
    "matrix": index_matrix,
    "lora_paths": {k: lora_index[k]["lora_path"] for k in index_keys}
}
with open("lora_index.pkl", "wb") as f:
    pickle.dump(index_save, f)
print("Index saved to lora_index.pkl")

# To reload later:
# with open("lora_index.pkl", "rb") as f:
#     index_save = pickle.load(f)
# index_keys, index_matrix = index_save["keys"], index_save["matrix"]